In [86]:
import os
import pandas as pd
import numpy as np
import sqlite3

In [87]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DB_PATH = os.path.join(BASE_DIR, "2_DATABASE", "eko_cards.db")
PRICES_PATH = os.path.join(BASE_DIR, "3_PROJECT_DATA", "prices.xlsx")

In [88]:
def transaction_import():

    # -------------------------
    # READ EXCEL
    # -------------------------
    data = pd.read_excel(
        os.path.join(BASE_DIR, "3_PROJECT_DATA", "eko_transactions.xlsx"),
        dtype={"Number": str}
    )

    # -------------------------
    # CLEAN COLUMN NAMES
    # -------------------------
    data.columns = data.columns.str.strip()

    # -------------------------
    # CLEAN MATERIAL
    # -------------------------
    data["Material"] = (
        data["Material"]
        .astype(str)
        .str.replace(r'^\d{6,}\s*', '', regex=True)   # remove product code
        .str.replace(r"\s+", " ", regex=True)      # normalize spaces
        .str.strip()
        .str.upper()                               # normalize text
    )

    # -------------------------
    # CLEAN VEHICLE
    # -------------------------
    data["Name"] = (
        data["Name"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

    # -------------------------
    # SELECT COLUMNS
    # -------------------------
    data = data[
        [
            "Plant",
            "Name",
            "Number",
            "Material",
            "Date",
            "Bill.qty",
            "Bill.qty.1",
            "FinPr",
            "Tot Amount",
            "Auth.time",
            "Km stand",
            "Billing Document"
        ]
    ]

    # -------------------------
    # FORMAT DATE
    # -------------------------
    data["Date"] = pd.to_datetime(
        data["Date"],
        dayfirst=True,
        errors="coerce"
    ).dt.date

    data = data[data["Date"].notna()]

    data["Date"] = data["Date"].astype(str)

    # -------------------------
    # CLEAN CARD NUMBERS
    # -------------------------
    data["Number"] = (
        data["Number"]
        .astype(str)
        .str.replace(r"\D", "", regex=True)
    )

    data = data[data["Number"].str.len() > 0]

    # -------------------------
    # REMOVE NON TRANSACTIONS
    # -------------------------
    data["Bill.qty"] = pd.to_numeric(data["Bill.qty"], errors="coerce")
    data = data[data["Bill.qty"] > 0]

    # -------------------------
    # RENAME COLUMNS
    # -------------------------
    data = data.rename(columns={
        "Plant": "plant",
        "Name": "name",
        "Number": "card_number",
        "Material": "material",
        "Date": "date",
        "Bill.qty": "bill_qty",
        "Bill.qty.1": "bill_qty2",
        "FinPr": "price",
        "Tot Amount": "amount",
        "Auth.time": "auth_time",
        "Km stand": "Km_stand",
        "Billing Document": "Billing_Document"
    })
    data.columns = data.columns.str.strip()
    # -------------------------
    # CONNECT 2_DATABASE
    # -------------------------
    conn = sqlite3.connect(DB_PATH)

    data.to_sql(
        "transactions",
        conn,
        if_exists="append",
        index=False
    )

    conn.close()

    print(f"Transactions imported successfully: {len(data)} rows")

In [89]:
def price_import():
    # -------------------------
    # READ
    # -------------------------
    prices = pd.read_excel(PRICES_PATH)
    prices.columns = prices.columns.str.strip()

    prices = prices[["ДАТА", "ФИРМА", "ЕИК", "ПРОДУКТ", "МАРЖ", "КРАЙНА_ЦЕНА", "ОТСТЪПКА"]]

    prices = prices.rename(columns={
        "ДАТА": "date",
        "ФИРМА": "company",
        "ЕИК": "eik",
        "ПРОДУКТ": "product",
        "МАРЖ": "margin",
        "КРАЙНА_ЦЕНА": "final_price",
        "ОТСТЪПКА": "discount"
    })

    # -------------------------
    # CLEAN
    # -------------------------
    prices["company"] = (
        prices["company"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

    prices["product"] = (
        prices["product"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

    prices["eik"] = prices["eik"].astype(str).str.strip()

    prices["date"] = pd.to_datetime(prices["date"], dayfirst=True, errors="coerce")
    prices = prices[prices["date"].notna()]
    prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")

    prices["final_price"] = pd.to_numeric(prices["final_price"], errors="coerce")
    prices = prices[prices["final_price"].notna()]
    prices["margin"] = pd.to_numeric(prices["margin"], errors="coerce")
    prices["discount"] = pd.to_numeric(prices["discount"], errors="coerce").fillna(0.0)

    # -------------------------
    # DB
    # -------------------------
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print("DB PATH:", DB_PATH)

    # check table
    table_info = pd.read_sql("PRAGMA table_info(prices)", conn)
    print("TABLE STRUCTURE BEFORE:")
    print(table_info)

    # add column if missing
    if "eik" not in table_info["name"].values:
        print("Adding eik column...")
        cursor.execute("ALTER TABLE prices ADD COLUMN eik TEXT")
        conn.commit()


    if "margin" not in table_info["name"].values:
        print("Adding margin column...")
        cursor.execute("ALTER TABLE prices ADD COLUMN margin REAL")
        conn.commit()

    if "discount" not in table_info["name"].values:
        print("Adding discount column...")
        cursor.execute("ALTER TABLE prices ADD COLUMN discount REAL")
        conn.commit()

    # verify again
    table_info = pd.read_sql("PRAGMA table_info(prices)", conn)
    print("TABLE STRUCTURE AFTER:")
    print(table_info)

    # -------------------------
    # INSERT (SAFE)
    # -------------------------
    insert_query = """
                   INSERT INTO prices (date, company, product, final_price, eik, margin, discount)
                   VALUES (?, ?, ?, ?, ?, ?, ?)
                   """

    data = prices[["date", "company", "product", "final_price", "eik", "margin", "discount"]].values.tolist()

    print("SAMPLE DATA:")
    print(data[:6])

    cursor.executemany(insert_query, data)

    conn.commit()

    # -------------------------
    # VERIFY INSERT
    # -------------------------
    check = pd.read_sql("SELECT * FROM prices ORDER BY ROWID DESC LIMIT 6", conn)
    print("LAST INSERTED ROWS:")
    print(check)

    conn.close()

    print(f"Prices imported successfully: {len(data)} rows")

In [90]:
def company_import():

    cards = pd.read_excel("../3_PROJECT_DATA/cards.xlsx", dtype={"Number": str})

    cards = cards.loc[:, ~cards.columns.str.contains('^Unnamed')]

    cards.columns = cards.columns.str.strip()

    cards = cards.rename(columns={
    "Company": "company",
    "Name": "vehicle",
    "Number": "card_number",
    "EIK": "eik"
    })


    cards["company"] = cards["company"].astype(str).str.strip()

    conn = sqlite3.connect(DB_PATH)

    cards.to_sql(
        "cards",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

In [91]:
def merge_tables():
    import sqlite3
    import pandas as pd

    conn = sqlite3.connect(DB_PATH)

    query = """
    SELECT
        t.*,
        c.company,
        p.margin,
        p.discount,
        p.final_price AS base_price,

        CASE
            WHEN COALESCE(p.discount, 0) > 0 THEN
                CAST(
                    (
                        CASE
                            WHEN (t.price - COALESCE(p.discount, 0)) < 0 THEN 0
                            ELSE (t.price - COALESCE(p.discount, 0))
                        END
                    ) * 100 + 0.5
                AS INTEGER) / 100.0

            WHEN p.final_price IS NOT NULL THEN
                CAST(p.final_price * 100 + 0.5 AS INTEGER) / 100.0

            ELSE
                CAST(t.price * 100 + 0.5 AS INTEGER) / 100.0
        END AS final_price

    FROM transactions t

    LEFT JOIN cards c
        ON t.card_number = c.card_number

    LEFT JOIN prices p
        ON p.eik = c.eik
        AND UPPER(p.product) = UPPER(t.material)
        AND p.date = (
            SELECT MAX(p2.date)
            FROM prices p2
            WHERE p2.eik = c.eik
              AND UPPER(p2.product) = UPPER(t.material)
              AND p2.date <= t.date
        )
        AND p.rowid = (
            SELECT p3.rowid
            FROM prices p3
            WHERE p3.eik = c.eik
              AND UPPER(p3.product) = UPPER(t.material)
              AND p3.date = (
                  SELECT MAX(p4.date)
                  FROM prices p4
                  WHERE p4.eik = c.eik
                    AND UPPER(p4.product) = UPPER(t.material)
                    AND p4.date <= t.date
              )
            ORDER BY p3.rowid DESC
            LIMIT 1
        )
    """

    data = pd.read_sql(query, conn)

    return data

In [92]:
def generate_result(data):

    os.makedirs(os.path.join(BASE_DIR, "A_FINAL_RESULT"), exist_ok=True)

    output_file = os.path.join(BASE_DIR, "A_FINAL_RESULT", "eko_transactions.xlsx")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for company, df in data.groupby("company", dropna=False):

            company_name = "Unknown" if pd.isna(company) else str(company)
            sheet_name = company_name[:31]

            # -------------------------
            # NUMERIC SAFETY
            # -------------------------
            df["price"] = pd.to_numeric(df["price"], errors="coerce").round(2)
            df["final_price"] = pd.to_numeric(df["final_price"], errors="coerce").round(2)
            df["base_price"] = pd.to_numeric(df["base_price"], errors="coerce")
            df["bill_qty"] = pd.to_numeric(df["bill_qty"], errors="coerce").fillna(0)
            df["margin"] = pd.to_numeric(df["margin"], errors="coerce").fillna(0)

            if "discount" in df.columns:
                df["discount"] = pd.to_numeric(df["discount"], errors="coerce").fillna(0).round(3)
            else:
                df["discount"] = 0.0

            # -------------------------
            # TOTALS
            # -------------------------
            df["eko_total"] = (df["bill_qty"] * df["price"]).round(2)
            df["gta_total"] = (df["bill_qty"] * df["final_price"]).round(2)

            # -------------------------
            # PROFIT
            # -------------------------
            df["profit"] = np.where(
                df["margin"] > 0,
                df["bill_qty"] * df["margin"],
                (df["final_price"] - df["base_price"]) * df["bill_qty"]
            )

            df["profit"] = pd.to_numeric(df["profit"], errors="coerce").round(2)

            # -------------------------
            # COLUMN ORDER
            # -------------------------
            df = df[
                [
                    "date",
                    "plant",
                    "name",
                    "card_number",
                    "material",
                    "bill_qty",
                    "bill_qty2",
                    "final_price",
                    "discount",
                    "margin",
                    "profit",
                    "gta_total",
                    "price",
                    "eko_total",
                    "auth_time",
                    "Km_stand",
                    "Billing_Document"
                ]
            ]

            # -------------------------
            # RENAME
            # -------------------------
            df = df.rename(columns={
                "date": "Дата",
                "plant": "Станция",
                "name": "Име",
                "card_number": "Номер на карта",
                "material": "Име на артикул",
                "bill_qty": "Литри",
                "bill_qty2": "Тип количество",
                "final_price": "GTA цена",
                "discount": "Отстъпка",
                "margin": "Марж",
                "profit": "Печалба",
                "gta_total": "Сума по GTA цена",
                "price": "ЕКО цена",
                "eko_total": "Сума по ЕКО цена",
                "auth_time": "Час",
                "Km_stand": "Километри",
                "Billing_Document": "Billing_Document"
            })

            # -------------------------
            # COMPANY HEADER
            # -------------------------
            header_df = pd.DataFrame([[f"Наименование на дружеството: {company_name}"]])

            header_df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                header=False,
                startrow=0
            )

            # -------------------------
            # DATA
            # -------------------------
            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                startrow=2
            )


    print("Output file generated:", output_file)

In [93]:
def delete_transactions():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM transactions")

    conn.commit()
    conn.close()

In [94]:
def delete_company():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM cards")

    conn.commit()
    conn.close()

In [95]:
def delete_prices():

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM prices")

    conn.commit()
    conn.close()

    print("All prices deleted.")

In [96]:
def delete_prices_by_month(year, month):

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    month_str = f"{year}-{str(month).zfill(2)}"

    cursor.execute(
        """
        DELETE FROM prices
        WHERE strftime('%Y-%m', date) = ?
        """,
        (month_str,)
    )

    conn.commit()
    conn.close()

    print(f"Prices deleted for: {month_str}")

In [97]:
def export_transactions_to_excel():

    import sqlite3
    import pandas as pd
    import os

    conn = sqlite3.connect(DB_PATH)

    df = pd.read_sql("SELECT * FROM transactions", conn)

    conn.close()

    # същата логика като generate_result
    os.makedirs(os.path.join(BASE_DIR, "A_FINAL_RESULT"), exist_ok=True)

    output_file = os.path.join(BASE_DIR, "A_FINAL_RESULT", "transactions_export.xlsx")

    df.to_excel(output_file, index=False)

    print(f"Export successful: {output_file}")

In [98]:
def normalize_text(text):
    if not text:
        return text

    replacements = {
        'Е': 'E',
        'е': 'e',
        'А': 'A',
        'а': 'a',
        'О': 'O',
        'о': 'o',
        'Р': 'P',
        'р': 'p',
        'С': 'C',
        'с': 'c',
        'М': 'M',
        'м': 'm',
        'Т': 'T',
        'т': 't',
        'В': 'B',
        'в': 'b',
        'Н': 'H',
        'н': 'h',
        'К': 'K',
        'к': 'k',
        'Х': 'X',
        'х': 'x',
    }

    for cyr, lat in replacements.items():
        text = text.replace(cyr, lat)

    return text

In [99]:
def normalize_db_text():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("SELECT id, material FROM transactions")
    rows = cursor.fetchall()

    for row_id, material in rows:
        normalized = normalize_text(material)

        if normalized != material:
            cursor.execute(
                "UPDATE transactions SET material = ? WHERE id = ?",
                (normalized, row_id)
            )

    conn.commit()
    conn.close()

In [100]:
def normalize_prices():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("SELECT id, product FROM prices")
    rows = cursor.fetchall()

    for row_id, product in rows:
        normalized = normalize_text(product)

        if normalized != product:
            cursor.execute(
                "UPDATE prices SET product = ? WHERE id = ?",
                (normalized, row_id)
            )

    conn.commit()
    conn.close()

In [101]:
# =========================
# PIPELINE
# =========================
def execute_pipeline():

    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM transactions")
    conn.commit()
    conn.close()

    transaction_import()

    normalize_db_text()
    normalize_prices()

    data = merge_tables()
    generate_result(data)